# Auto Math Research Demo

This notebook demonstrates the interaction with the research database and the optimization loop.

In [ ]:
import sqlite3
import pandas as pd
import sys
from pathlib import Path

# Ensure project root is in path
sys.path.append('auto-math-research')

from configs.research_config import ResearchConfig

DB_PATH = ResearchConfig.DB_PATH
print(f"Connected to database: {DB_PATH}")

In [ ]:
def get_top_optimizers(limit=10):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(
        """
        SELECT id, generation, objective_value, status, created_at, latex_formula 
        FROM optimizers 
        ORDER BY objective_value DESC 
        LIMIT ?
        """, 
        conn, 
        params=(limit,)
    )
    conn.close()
    return df

get_top_optimizers()

## Simulate Mutator Step
Run the mutator for one iteration to generate a new candidate.

In [ ]:
from optimizers.mutator import get_best_parent, mutate_via_llm, insert_candidate
import json

conn = sqlite3.connect(DB_PATH)
parent = get_best_parent(conn)

if parent:
    print(f"Found parent ID: {parent['id']} Gen: {parent['generation']}")
    parent_ast = json.loads(parent["canonical_ast_json"])
    
    try:
        candidate = mutate_via_llm(parent_ast)
        inserted = insert_candidate(
            conn,
            parent_id=parent["id"],
            generation=parent["generation"] + 1,
            raw_ast=candidate,
            latex_formula="auto-generated"
        )
        if inserted:
            print(f"Inserted new candidate for generation {parent['generation'] + 1}")
        else:
            print("Duplicate candidate generated, skipped.")
    except Exception as e:
        print(f"Error during mutation: {e}")
else:
    print("No parent found to mutate from.")
    
conn.close()

## Simulate Worker Step
Run a worker to process the next pending task.

In [ ]:
from training.worker import claim_next_pending, mark_done, _worker_id
from configs.research_config import ResearchConfig
import random

worker_id = _worker_id("notebook-demo")
conn = sqlite3.connect(DB_PATH)

task = claim_next_pending(conn, worker_id)

if task:
    print(f"Claimed task ID: {task['id']}")
    print(f"AST: {task['raw_ast_json']}")
    
    # Simulate work
    # Here you would normally execute the training loop
    
    # Generate dummy metrics
    simulated_loss = random.uniform(0.1, 10.0)
    simulated_objective = 1.0 / simulated_loss
    flops_estimate = 1e9
    
    mark_done(conn, task['id'], simulated_objective, simulated_loss, flops_estimate, ResearchConfig.DEFAULT_STEPS)
    print(f"Task {task['id']} completed with objective {simulated_objective:.4f}")
else:
    print("No pending tasks found.")

conn.close()

In [ ]:
# Check status again
get_top_optimizers()